# FAC_NB_Leer_Catalogo — variante alternativa para leer el catálogo desde el pipeline

Devuelve, como **JSON por `exitValue`**, las filas **activas** del catálogo `fac_catalogo_ingesta`, ya **ordenadas** por `orden` (y filtradas por `filtro_tabla` si se indica).

## ¿Para qué existe si el pipeline principal ya usa una actividad Lookup?

El pipeline principal lee el catálogo con una actividad **Lookup** sobre la tabla del Lakehouse en modo *Tabla* (lee el Delta directamente). Ese modo no admite `WHERE` ni `ORDER BY`. La alternativa con consulta T-SQL pasa por el **endpoint SQL del Lakehouse**, que puede tardar en reflejar tablas y cambios recién escritos por Spark (justo lo que hace esta demo en cada corrida) y requiere una conexión creada aparte. Este cuaderno lee el Delta con Spark: **sin latencia de sincronización**, con filtro y orden garantizados, y con una validación explícita si el filtro no coincide con nada.

Uso en el pipeline: actividad Notebook `Leer_Catalogo` → `@json(activity('Leer_Catalogo').output.result.exitValue).tablas` como *Items* del ForEach. Ver `FAC_PL_Ingesta_Metadata.alt-notebook.json`.

In [ ]:
# Celda de parámetros
filtro_tabla = ""      # vacío = todas las tablas activas; o el nombre exacto de una tabla destino
destino = "auto"       # auto | ninguno | nombre de esquema

## Leer, filtrar, ordenar y devolver

Se usa la API de DataFrame (no se concatena el filtro en SQL). Si el filtro no coincide con ninguna tabla activa, el cuaderno **falla con un mensaje claro** en lugar de dejar que el pipeline "termine bien" sin cargar nada.

In [ ]:
import json
import re
import uuid
import datetime
from pyspark.sql import functions as F

PATRON_IDENT = re.compile(r"[A-Za-z][A-Za-z0-9_]{0,63}\Z")


def validar_ident(valor, campo):
    """Los nombres se interpolan en SQL: solo se aceptan identificadores simples."""
    if not isinstance(valor, str) or PATRON_IDENT.fullmatch(valor) is None:
        raise ValueError(
            f"{campo}: se esperaba un identificador (letra inicial; letras, números y _); se recibió {valor!r}"
        )
    return valor


def resolver_esquema(destino):
    """auto = 'fac' si el Lakehouse tiene esquemas habilitados, si no ninguno; 'ninguno' o un nombre explícito."""
    d = (destino or "auto").strip()
    if d.lower() == "ninguno":
        return ""
    if d.lower() == "auto":
        try:
            nombres = {str(r[0]).lower() for r in spark.sql("SHOW SCHEMAS").collect()}
        except Exception:
            nombres = set()
        return "fac" if "dbo" in nombres else ""
    return validar_ident(d, "destino")


def calificar(esquema, tabla):
    return f"{esquema}.{tabla}" if esquema else tabla

import notebookutils

filtro_tabla = (filtro_tabla or "").strip()
if filtro_tabla:
    validar_ident(filtro_tabla, "filtro_tabla")

ESQUEMA = resolver_esquema(destino)
CAT = calificar(ESQUEMA, "fac_catalogo_ingesta")
if not spark.catalog.tableExists(CAT):
    raise RuntimeError(f"No existe {CAT}. Ejecuta primero FAC_NB_Preparar_Control.")

df = spark.table(CAT).where(F.col("activo") == 1)
if filtro_tabla:
    df = df.where(F.col("tabla_destino") == filtro_tabla)

filas = []
for r in df.orderBy("orden", "id").collect():
    d = r.asDict()
    filas.append({k: ("" if v is None else v) for k, v in d.items()})

if not filas:
    raise RuntimeError("No hay tablas activas que coincidan" + (f" con filtro_tabla={filtro_tabla!r}" if filtro_tabla else ""))

display(df.orderBy("orden", "id"))
salida = {"total": len(filas), "tablas": filas}
print(json.dumps(salida, ensure_ascii=False)[:2000])

## Salir con el JSON

`exit()` va en su propia celda y fuera de cualquier `try/except` (dentro de uno no surte efecto). El valor de salida es siempre **texto**; el pipeline lo convierte con `@json(...)`.

In [ ]:
notebookutils.notebook.exit(json.dumps(salida, ensure_ascii=False))